#### 향후 pcd-voxel-STEP 데이터셋 구축 시:
- pcd + camera viewpoint / voxel / B-Rep 정보 저장 필요

In [9]:
import os, random
import numpy as np
import open3d as o3d
from pathlib import Path
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopAbs import TopAbs_FACE
from OCC.Core.BRep import BRep_Tool
from OCC.Core.TopLoc import TopLoc_Location

## STEP to Point cloud, Occlusion by Random viewpoint, Sensor Noise

In [10]:
############ 모델 불러오기 ############
# 경로 설정
base_dir = "./abc_dataset_filtered-1"
prefix_folder_ind = 0 # 0~99
prefix_folder = str(prefix_folder_ind).zfill(4)
prefix_path = Path(base_dir) / prefix_folder

model_id_list = os.listdir(prefix_path) # 00xx0000~00xx9999 
# model_id = random.choice(model_id_list)
model_id = prefix_folder + '0006'

# 경로 조합
model_path = Path(prefix_path) / model_id
step_path = list(model_path.glob('*.step'))[0]

print(step_path)

abc_dataset_filtered-1\0000\00000006\00000006_d4fe04f0f5f84b52bd4f10e4_step_001.step


In [11]:
def load_step_as_mesh(step_path):
    """STEP 파일을 읽어 변환 행렬이 적용된 Open3D Mesh로 변환"""
    str_path = os.path.abspath(str(step_path))
    reader = STEPControl_Reader()
    if reader.ReadFile(str_path) != 1:
        raise Exception("STEP 파일을 읽을 수 없습니다.")
    
    reader.TransferRoots()
    shape = reader.OneShape()
    BRepMesh_IncrementalMesh(shape, 0.1).Perform()
    
    verts, tris = [], []
    explorer = TopExp_Explorer(shape, TopAbs_FACE)
    while explorer.More():
        face = explorer.Current()
        loc = TopLoc_Location()
        tri = BRep_Tool.Triangulation(face, loc)
        if tri:
            v_offset = len(verts)
            trans = loc.Transformation()
            for i in range(1, tri.NbNodes() + 1):
                p = tri.Node(i).Transformed(trans)
                verts.append([p.X(), p.Y(), p.Z()])
            for i in range(1, tri.NbTriangles() + 1):
                t = tri.Triangle(i)
                tris.append([t.Value(1)-1+v_offset, t.Value(2)-1+v_offset, t.Value(3)-1+v_offset])
        explorer.Next()
    
    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(np.array(verts))
    mesh.triangles = o3d.utility.Vector3iVector(np.array(tris))
    mesh = mesh.remove_duplicated_vertices()
    mesh.compute_vertex_normals()
    return mesh

def get_random_camera_pose(dist_range=(400, 800)):
    """원점을 바라보는 랜덤 구면 좌표계 카메라 위치 생성"""
    phi = np.random.uniform(0, 2 * np.pi)
    theta = np.arccos(np.random.uniform(-1, 1))
    radius = np.random.uniform(dist_range[0], dist_range[1])
    
    x = radius * np.sin(theta) * np.cos(phi)
    y = radius * np.sin(theta) * np.sin(phi)
    z = radius * np.cos(theta)
    return np.array([x, y, z])

def apply_advanced_simulation_raycasting(mesh, cam_pos, resolution=(640, 480)):
    """Raycasting 기반 Occlusion + 스테리오 Depth카메라 노이즈 시뮬레이션"""
    device = o3d.core.Device("CPU:0")
    scene = o3d.t.geometry.RaycastingScene()
    t_mesh = o3d.t.geometry.TriangleMesh.from_legacy(mesh, device=device)
    scene.add_triangles(t_mesh)

    view_vec = -cam_pos / np.linalg.norm(cam_pos)
    rays = scene.create_rays_pinhole(
        fov_deg=69, center=[0, 0, 0], eye=cam_pos,
        up=[0, 1, 0] if abs(view_vec[1]) < 0.9 else [1, 0, 0],
        width_px=resolution[0], height_px=resolution[1]
    )

    ans = scene.cast_rays(rays)
    t_hit = ans['t_hit'].numpy()
    hit_mask = np.isfinite(t_hit)
    
    # 좌표 계산: P = O + t*D 
    rays_arr = rays.numpy() # [H, W, 6] -> origin(3), direction(3)
    hit_origins = rays_arr[hit_mask][:, :3]
    hit_directions = rays_arr[hit_mask][:, 3:]
    hit_dist = t_hit[hit_mask]
    
    # 좌표 계산 공식: P = O + t * D
    hit_points = hit_origins + hit_dist[:, np.newaxis] * hit_directions

    sim_pcd = o3d.geometry.PointCloud()
    if len(hit_points) > 0:
        z_depths = np.dot(hit_points - cam_pos, view_vec)
        f, B, subpixel = 800, 50, 0.08 # d435i 스펙
        sigma = (z_depths**2 * subpixel) / (f * B) # d435 노이즈 수식
        
        noise = np.random.normal(0, 1, size=hit_points.shape)
        noise[:, 2] *= sigma # Z-Noise
        noise[:, 0:2] *= (sigma[:, np.newaxis] * 0.1)
        
        sim_pcd.points = o3d.utility.Vector3dVector(hit_points + noise)
    
    return sim_pcd

def get_fixed_voxel_input(pcd, resolution=64):
    """
    PCD를 Unit Sphere 정규화 후 고정 해상도 복셀 행렬로 변환 (AI 입력용)
    """
    if pcd.is_empty():
        return np.zeros((resolution, resolution, resolution)), 1.0

    points = np.asarray(pcd.points).copy()
    
    # 1. 정규화 (Normalization): 모든 점을 [-0.5, 0.5] 공간으로 스케일링
    # (이미 mesh.translate(-center)가 되어 있다고 가정)
    max_dist = np.max(np.linalg.norm(points, axis=1))
    # 안전을 위해 약간의 여유(0.9)를 두고 스케일링
    points = (points / max_dist) * 0.45 
    
    # 2. 복셀 인덱스로 변환: [-0.5, 0.5] -> [0, resolution-1]
    voxel_indices = ((points + 0.5) * (resolution - 1)).astype(int)
    
    # 3. 3D Occupancy Grid 생성
    voxel_matrix = np.zeros((resolution, resolution, resolution), dtype=np.float32)
    voxel_matrix[voxel_indices[:, 0], voxel_indices[:, 1], voxel_indices[:, 2]] = 1.0
    
    # 4. 시각화용 Open3D 객체 생성
    indices = np.argwhere(voxel_matrix == 1.0)
    v_pcd = o3d.geometry.PointCloud()
    v_pcd.points = o3d.utility.Vector3dVector(indices)
    voxel_grid = o3d.geometry.VoxelGrid.create_from_point_cloud(v_pcd, voxel_size=1.0)
    
    return voxel_matrix, voxel_grid

In [12]:
# 1. 메시 로드 및 중심화
mesh = load_step_as_mesh(step_path)
mesh.translate(-mesh.get_center())

# 2. 랜덤 시계 시뮬레이션
cam_pos = get_random_camera_pose(dist_range=(300, 800))
sim_pcd = apply_advanced_simulation_raycasting(mesh, cam_pos)

# 3. 고정 해상도 복셀 추출
vox_matrix, vox_grid = get_fixed_voxel_input(sim_pcd, resolution=64)

print(f"추출된 포인트 수: {len(sim_pcd.points)}")
print(f"입력 텐서(Voxel Matrix) 크기: {vox_matrix.shape}")

# 4. 시각화 데이터 전처리
voxels = vox_grid.get_voxels()
vox_pcd_viz = o3d.geometry.PointCloud()
# grid_index 리스트를 numpy array(float64)로 변환
indices = np.array([v.grid_index for v in voxels]).astype(np.float64)
vox_pcd_viz.points = o3d.utility.Vector3dVector(indices)
vox_pcd_viz.paint_uniform_color([0, 1, 0]) # 복셀은 초록색
vox_pcd_viz.translate([150, 0, 0]) # 메쉬와 겹치지 않게 옆으로 이동

sim_pcd.paint_uniform_color([1, 0, 0])      # PCD 빨간색
mesh.paint_uniform_color([0.7, 0.7, 0.7])   # 메쉬 회색

# 5. 시각화
print(" - [=] / [-] : 점 크기 확대/축소")
print(" - [L] : 라이팅 온/오프")
print(" - [K] : 배경색 변경 (검정/회색/흰색)")
print(" - [W] : 와이어프레임 모드 토글")

o3d.visualization.draw_geometries(
    [mesh, sim_pcd, vox_pcd_viz], 
    window_name="PCD & Voxel Simulation - draw_geometries",
    width=1024,
    height=768,
    left=50,
    top=50,
    mesh_show_back_face=True # 이 인자는 버전따라 지원 여부가 다를 수 있음
)

추출된 포인트 수: 5
입력 텐서(Voxel Matrix) 크기: (64, 64, 64)
 - [=] / [-] : 점 크기 확대/축소
 - [L] : 라이팅 온/오프
 - [K] : 배경색 변경 (검정/회색/흰색)
 - [W] : 와이어프레임 모드 토글
